In [1]:
import pandas as pd
import pm4py
from collections import Counter



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
df = pd.read_csv("actual_event_log.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

,case_id,activity,timestamp
0,P00001,Registration,2026-10-07 20:32:41+00:00
1,P00001,Triage,2026-10-07 20:42:41+00:00
2,P00001,Lab Test,2026-10-07 20:49:41+00:00
3,P00001,Treatment,2026-10-07 21:12:41+00:00
4,P00001,Doctor Consultation,2026-10-07 21:25:41+00:00


In [3]:
#Convert the data into a PM4Py event log
df = df.sort_values(["case_id", "timestamp"])

event_log = pm4py.format_dataframe(
    df,
    case_id="case_id",
    activity_key="activity",
    timestamp_key="timestamp"
)

event_log = pm4py.convert_to_event_log(event_log)

print("Total patient cases:", len(event_log))

Total patient cases: 5000


In [5]:
#Load your ideal Petri net
net, initial_marking, final_marking = pm4py.read_pnml(
    "ideal_reference_model.pnml"
)

print("Ideal reference model loaded successfully")

Ideal reference model loaded successfully


In [6]:
#Calculate conformance fitness
fitness = pm4py.fitness_alignments(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Alignment fitness results:")
print(fitness)

aligning log, completed variants ::   0%|          | 0/18 [00:00<?, ?it/s]

Alignment fitness results:
{'percFitTraces': 20.88, 'averageFitness': 0.8848767909867811, 'percentage_of_fitting_traces': 20.88, 'average_trace_fitness': 0.8848767909867811, 'log_fitness': 0.8809347550671689}


In [7]:
#Now calculate token-based replay fitness as a second measure:
token_fitness = pm4py.fitness_token_based_replay(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Token-based fitness results:")
print(token_fitness)

replaying log with TBR, completed traces ::   0%|          | 0/18 [00:00<?, ?it/s]

Token-based fitness results:
{'perc_fit_traces': 20.88, 'average_trace_fitness': 0.8531504761904762, 'log_fitness': 0.8632916244558224, 'percentage_of_fitting_traces': 20.88}


In [8]:
#Identify deviations in individual patient journeys
diagnostics = pm4py.conformance_diagnostics_alignments(
    event_log,
    net,
    initial_marking,
    final_marking
)

print("Number of patient cases analyzed:", len(diagnostics))
print("Example alignment:")
print(diagnostics[0])

aligning log, completed variants ::   0%|          | 0/18 [00:00<?, ?it/s]

Number of patient cases analyzed: 5000
Example alignment:
{'alignment': [('Registration', 'Registration'), ('Triage', 'Triage'), ('Lab Test', 'Lab Test'), ('>>', None), ('>>', None), ('Treatment', 'Treatment'), ('Doctor Consultation', 'Doctor Consultation'), ('Discharge', 'Discharge')], 'cost': 2, 'visited_states': 8, 'queued_states': 26, 'traversed_arcs': 26, 'lp_solved': 2, 'fitness': 1.0, 'bwc': 110003}


In [9]:
#Create a patient-level deviation summary
deviation_rows = []

for case_id, alignment_result in zip(
    [trace.attributes["concept:name"] for trace in event_log],
    diagnostics
):
    alignment = alignment_result.get("alignment", [])

    log_moves = []
    model_moves = []

    for log_move, model_move in alignment:
        if log_move is not None and model_move is None:
            log_moves.append(log_move)

        elif log_move is None and model_move is not None:
            model_moves.append(model_move)

    deviation_rows.append({
        "case_id": case_id,
        "log_deviation_count": len(log_moves),
        "model_deviation_count": len(model_moves),
        "total_deviations": len(log_moves) + len(model_moves),
        "log_deviations": ", ".join(map(str, log_moves)),
        "missing_model_activities": ", ".join(map(str, model_moves))
    })

deviation_df = pd.DataFrame(deviation_rows)

deviation_df.head(10)

,case_id,log_deviation_count,model_deviation_count,total_deviations,log_deviations,missing_model_activities
0,P00001,2,0,2,">>, >>",
1,P00002,2,0,2,">>, >>",
2,P00003,2,0,2,">>, >>",
3,P00004,2,0,2,">>, >>",
4,P00005,2,0,2,">>, >>",
5,P00006,2,0,2,">>, >>",
6,P00007,2,0,2,">>, >>",
7,P00008,3,0,3,">>, >>, >>",
8,P00009,2,0,2,">>, >>",
9,P00010,2,0,2,">>, >>",


In [10]:
#Find the most common deviations
deviation_summary = (
    deviation_df.groupby(
        ["log_deviations", "missing_model_activities"],
        dropna=False
    )
    .size()
    .reset_index(name="patient_count")
    .sort_values("patient_count", ascending=False)
)

deviation_summary.head(20)

,log_deviations,missing_model_activities,patient_count
0,">>, >>",,3879
1,">>, >>, >>",,1121


In [11]:
#calculate how many patient cases have at least one deviation:
total_cases = len(deviation_df)

deviated_cases = (
    deviation_df["total_deviations"] > 0
).sum()

compliant_cases = total_cases - deviated_cases

print("Total patient cases:", total_cases)
print("Cases with deviations:", deviated_cases)
print("Cases without detected deviations:", compliant_cases)

print(
    "Deviation rate:",
    round(deviated_cases / total_cases * 100, 2),
    "%"
)

Total patient cases: 5000
Cases with deviations: 5000
Cases without detected deviations: 0
Deviation rate: 100.0 %


In [12]:
#Save your results
deviation_df.to_csv(
    "patient_level_deviations.csv",
    index=False
)

deviation_summary.to_csv(
    "deviation_summary.csv",
    index=False
)

print("Part 4 results saved successfully!")

Part 4 results saved successfully!
